# Episodi simulati per ACT — SO-101, caramelle

500 episodi **riusciti** (i falliti si scartano e restano nel registro col motivo),
4 processi in parallelo con semi diversi, poi un merge solo.
Codice: `robotica` (dataset `so101-sim-codice`), stesso commit del portatile.
Output: `caramelle_sim/` (dataset LeRobot) e i registri `*_generazione.jsonl`.

In [ ]:
import time; INIZIO = time.time()
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv; nproc; free -g | head -2; df -h /tmp /kaggle/working | tail -2

In [ ]:
import glob, os, shutil
s = glob.glob('/kaggle/input/**/robotica/scripts/genera_act_sim.py', recursive=True)
assert s, 'pacchetto codice non trovato: e\' in Input?'
shutil.copytree(os.path.dirname(os.path.dirname(s[0])), '/tmp/robotica', dirs_exist_ok=True)
print('codice:', s[0])

In [ ]:
!pip install -q uv
!UV_CACHE_DIR=/tmp/uvcache uv venv -q -p 3.12 /tmp/venv
!UV_CACHE_DIR=/tmp/uvcache uv pip install -q --python /tmp/venv/bin/python --torch-backend=cpu "lerobot[dataset]==0.6.1" "mujoco==3.3.7" scipy
!/tmp/venv/bin/python -c "import lerobot, mujoco, numpy; print('lerobot', lerobot.__version__, '| mujoco', mujoco.__version__, '| numpy', numpy.__version__)"

## Chi disegna?

Il contenitore ha EGL solo di Mesa: senza `libEGL_nvidia`, "egl" disegna sulla
CPU (llvmpipe, 181 ms a fotogramma) senza dire niente. La T4 vera: 1,3 ms.
Le librerie GL di NVIDIA devono avere la versione **esatta** del driver: da apt
se coincide, altrimenti dall'installer NVIDIA estratto in una cartella privata.
Poi si disegna la **nostra** scena a meta' episodio e si esige NVIDIA e
un'immagine non uniforme. Un render che non da' errore non prova niente.

In [ ]:
import subprocess, os, json
ver = subprocess.run(['nvidia-smi', '--query-gpu=driver_version', '--format=csv,noheader'], capture_output=True, text=True).stdout.split()[0]
subprocess.run('apt-get -qq update > /dev/null 2>&1', shell=True)
cand = subprocess.run(f"apt-cache policy libnvidia-gl-{ver.split('.')[0]} | grep Candidate", shell=True, capture_output=True, text=True).stdout
print('driver', ver, '| apt', cand.strip())
AMB = {**os.environ, 'MUJOCO_GL': 'egl', 'PYOPENGL_PLATFORM': 'egl'}
if ver in cand:
    r = subprocess.run(f"apt-get -qq install -y --no-install-recommends libnvidia-gl-{ver.split('.')[0]}", shell=True, capture_output=True, text=True)
    print('apt esito', r.returncode, r.stderr[-300:])
    AMB['__EGL_VENDOR_LIBRARY_FILENAMES'] = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
else:  # piano B, provato nella diagnosi: stesse prestazioni
    for url in (f'https://us.download.nvidia.com/tesla/{ver}/NVIDIA-Linux-x86_64-{ver}.run',
                f'https://us.download.nvidia.com/XFree86/Linux-x86_64/{ver}/NVIDIA-Linux-x86_64-{ver}.run'):
        if subprocess.run(['wget', '-q', url, '-O', '/tmp/nv.run']).returncode == 0:
            break
    subprocess.run('sh /tmp/nv.run --extract-only --target /tmp/nvx > /dev/null && mkdir -p /tmp/nvlib', shell=True, check=True)
    for nome in ('libEGL_nvidia', 'libnvidia-eglcore', 'libnvidia-glsi', 'libnvidia-gpucomp'):
        if os.path.exists(f'/tmp/nvx/{nome}.so.{ver}'):
            shutil.copy(f'/tmp/nvx/{nome}.so.{ver}', '/tmp/nvlib/')
    os.symlink(f'libEGL_nvidia.so.{ver}', '/tmp/nvlib/libEGL_nvidia.so.0')
    json.dump({'file_format_version': '1.0.0', 'ICD': {'library_path': '/tmp/nvlib/libEGL_nvidia.so.0'}}, open('/tmp/nvlib/10_nvidia.json', 'w'))
    AMB['__EGL_VENDOR_LIBRARY_FILENAMES'] = '/tmp/nvlib/10_nvidia.json'
    AMB['LD_LIBRARY_PATH'] = '/tmp/nvlib:' + AMB.get('LD_LIBRARY_PATH', '')
    shutil.rmtree('/tmp/nvx'); os.remove('/tmp/nv.run')
r = subprocess.run(['/tmp/venv/bin/python', '/tmp/robotica/scripts/genera_act_sim.py', '/kaggle/working/prova', '--prova-render', '--gpu', '--seme', '7'],
                   env=AMB, capture_output=True, text=True)
print(r.stdout[-600:], r.stderr[-1500:] if r.returncode else '')
assert r.returncode == 0, 'il render non e\' sulla GPU o non disegna la scena'
from IPython.display import Image, display
display(Image('/kaggle/working/prova/prova_render.png'))

## Prova: 2 episodi a blocchi da 1, il loro merge, e la stima

Prova anche il meccanismo dei blocchi. Se la stima supera **10 h** ci si
ferma qui: meglio saperlo ora che dopo 12 h.

In [ ]:
t = time.time()
r = subprocess.run(['/tmp/venv/bin/python', '/tmp/robotica/scripts/genera_act_sim.py', '/tmp/prova/p', '--episodi', '2', '--seme', '7', '--blocco', '1'],
                   env=AMB, capture_output=True, text=True)
print(r.stdout[-800:], r.stderr[-1500:] if r.returncode else '')
assert r.returncode == 0
prova_s = time.time() - t
righe = [json.loads(x) for x in open('/tmp/prova/p_generazione.jsonl')]
for x in righe:
    print(x)
blocchi = sorted(glob.glob('/tmp/prova/p_b*'))
r = subprocess.run(['/tmp/venv/bin/lerobot-edit-dataset', '--new_repo_id', 'so101/prova', '--new_root', '/tmp/prova_unita',
                    '--operation.type', 'merge', '--operation.repo_ids', json.dumps([f'so101/b{i}' for i in range(len(blocchi))]),
                    '--operation.roots', json.dumps(blocchi)], capture_output=True, text=True)
info = json.load(open('/tmp/prova_unita/meta/info.json'))
print('blocchi', len(blocchi), '-> merge:', info['total_episodes'], 'episodi', info['total_frames'], 'fotogrammi')
assert len(blocchi) == 2 and info['total_episodes'] == 2, r.stderr[-800:]
# Un processo solo: con 4 insieme i core sono condivisi (fisica e video sono CPU), si conta 1,5x.
a_episodio = prova_s / 2
ore = a_episodio * 500 / 4 * 1.5 / 3600
print(f'{a_episodio:.0f} s a episodio scritto (falliti compresi) -> stima {ore:.1f} h per 500 con 4 processi')
assert ore < 10, f'stima {ore:.1f} h oltre 10: ci si ferma'
shutil.rmtree('/tmp/prova'); shutil.rmtree('/tmp/prova_unita')

## Tutti: 4 processi, 500 episodi, a blocchi da 25

Blocchi e registri direttamente in `/kaggle/working/blocchi`: un blocco chiuso
e' un dataset completo. Scadenza a 10.5 h dall'avvio del notebook: oltre,
nessun episodio nuovo, e si unisce quello che c'e'.

In [ ]:
import math
quanti = [math.ceil(500 / 4)] * 4
quanti[-1] = 500 - sum(quanti[:-1])
B = '/kaggle/working/blocchi'
os.makedirs(B, exist_ok=True)
scadenza = INIZIO + 10.5 * 3600
processi = []
for k, n in enumerate(quanti):
    log = open(f'{B}/parte_{k}.log', 'w')
    processi.append(subprocess.Popen(['/tmp/venv/bin/python', '/tmp/robotica/scripts/genera_act_sim.py', f'{B}/parte_{k}',
                                      '--episodi', str(n), '--seme', str(1_000_000 * (k + 1)), '--blocco', '25',
                                      '--scadenza', str(scadenza)], env=AMB, stdout=log, stderr=subprocess.STDOUT))
t0 = time.time()
while any(p.poll() is None for p in processi):
    time.sleep(300)
    fatti = provati = 0
    for k in range(len(quanti)):
        try:
            righe = [json.loads(x) for x in open(f'{B}/parte_{k}_generazione.jsonl')]
        except FileNotFoundError:
            continue
        provati += len(righe)
        fatti += sum('registra_s' in x and 'scartato' not in x for x in righe)
    ore = (time.time() - t0) / 3600
    fine = ore * 500 / fatti if fatti else float('nan')
    print(f'{ore * 60:.0f} min: {fatti} scritti su {provati} provati, fine prevista a {fine:.1f} h', flush=True)
for k, p in enumerate(processi):
    print('parte', k, 'esito', p.returncode, open(f'{B}/parte_{k}.log').read()[-300:])
assert all(p.returncode == 0 for p in processi)

## Un dataset solo, e i conti

In [ ]:
blocchi = sorted(glob.glob(f'{B}/parte_*_b*'))
blocchi = [b for b in blocchi if os.path.exists(b + '/meta/info.json')]
r = subprocess.run(['/tmp/venv/bin/lerobot-edit-dataset', '--new_repo_id', 'so101/caramelle_sim', '--new_root', '/tmp/caramelle_sim',
                    '--operation.type', 'merge', '--operation.repo_ids', json.dumps([f'so101/b{i}' for i in range(len(blocchi))]),
                    '--operation.roots', json.dumps(blocchi)], capture_output=True, text=True)
print(r.stdout[-800:], r.stderr[-800:])
info = json.load(open('/tmp/caramelle_sim/meta/info.json'))
attesi = sum(json.load(open(b + '/meta/info.json'))['total_episodes'] for b in blocchi)
print('blocchi', len(blocchi), '| episodi', info['total_episodes'], 'su', attesi, 'nei blocchi | fotogrammi', info['total_frames'])
assert info['total_episodes'] == attesi
if attesi < 500:
    print(f'ATTENZIONE: {attesi} episodi invece di 500 (scadenza)')
righe = [json.loads(x) for k in range(4) for x in open(f'{B}/parte_{k}_generazione.jsonl')]
scritti = [x for x in righe if 'registra_s' in x and 'scartato' not in x]
print(f'provati {len(righe)}, scritti {len(scritti)} ({100 * len(scritti) / len(righe):.0f}%), non deterministici {sum("scartato" in x for x in righe)}')
print('render ms/fotogramma mediana', sorted(x['render_ms'] for x in scritti)[len(scritti) // 2])
from collections import Counter
print('motivi di scarto:', Counter(m.split(' ')[1] if ' ' in m else m for x in righe for m in x.get('fallito', [])).most_common(6))
with open('/kaggle/working/generazione.jsonl', 'w') as f:
    f.writelines(json.dumps(x) + '\n' for x in righe)
# Solo ora, col merge verificato, si buttano i blocchi: lo spazio in /kaggle/working non raddoppia mai.
shutil.rmtree(B)
shutil.move('/tmp/caramelle_sim', '/kaggle/working/caramelle_sim')
print(f'fatto in {(time.time() - INIZIO) / 3600:.1f} h')